<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 22


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Project в C#, который будет представлять проекты в рамках
организации. На основе этого класса разработать 2-3 производных класса,
демонстрирующих принципы наследования и полиморфизма. В каждом из классов
должны быть реализованы новые атрибуты и методы, а также переопределены
некоторые методы базового класса для демонстрации полиморфизма. Добавьте к сущестующим классам конструктора классов с использованием гетторов и сетторов и реализуйте взаимодействие объектов между собой. Добавьте к существующим классам (базовым и производным 3-4 атрибута и метода) и реализуйте простое, сложное и множественное наследование. Добавьте к существующим классам (базовым и производным 3-4 атрибута и метода) и реализуйте полиморфизм с перекрытием и перегрузкой методов, а также generic классы. Добавьте к существующим классам (базовым и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями.

Требования к базовому классу Project:

• Атрибуты: ID проекта (ProjectId), Название проекта (ProjectName), Статус
проекта (Status).

• Методы:

o UpdateStatus(): метод для обновления статуса проекта.

o GetProjectDetails(): метод для получения деталей проекта.

o AssignTeamMember(): метод для назначения командного члена
проекту.

Требования к производным классам:
1. ПродуктовыйПроект (ProductProject): Должен содержать дополнительные
атрибуты, такие как Ожидаемая дата завершения (ExpectedCompletionDate).
Метод UpdateStatus() должен быть переопределен для включения
ожидаемой даты завершения в статус проекта.
2. ИсследовательскийПроект (ResearchProject): Должен содержать
дополнительные атрибуты, такие как Финансирование (FundingAmount).
Метод AssignTeamMember() должен быть переопределен для указания
специфических требований к навыкам командного члена для
исследовательского проекта.
3. ИнфраструктурныйПроект (InfrastructureProject) (если требуется третий
класс): Должен содержать дополнительные атрибуты, такие как Срок
выполнения (ExecutionPeriod). Метод GetProjectDetails() должен быть
переопределен для отображения срока выполнения проекта вместе с
другими деталями проекта.

#### Дополнительное задание
Добавьте к существующим классам (базовым и производным 3-4 атрибута и метода) используйте в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
public interface ILogger { void Log(string msg); }
public class ConsoleLogger : ILogger { public void Log(string msg) => Console.WriteLine(msg); }

public interface IExportable { string ExportToXml(); }
public interface IReportable { void GenerateReport(); }


public delegate void ProjectEventHandler(Project project, string action);

public class TeamMember {
    public string Name { get; set; }
    public string Skill { get; set; }
    public TeamMember(string name, string skill = "") { Name = name; Skill = skill; }
}


public class Project : IExportable {
    public int ProjectId { get; set; }
    public string ProjectName { get; set; }
    public string Status { get; set; }
    public decimal Budget { get; set; }
    public List<TeamMember> Team { get; set; } = new();

    
    public string Priority { get; set; } = "Средний";
    public int RiskLevel { get; set; } = 1;
    public DateTime StartDate { get; set; } = DateTime.Now;
    public HashSet<string> Tags { get; set; } = new();   

    
    public event ProjectEventHandler ProjectChanged;

    protected readonly ILogger _logger;   

    public Project(int id, string name, string status, ILogger logger, decimal budget = 0) {
        ProjectId = id; ProjectName = name; Status = status;
        Budget = budget; _logger = logger;
    }

    
    public virtual void UpdateStatus(string status) {
        Status = status;
        _logger.Log($"'{ProjectName}': статус -> {Status}");
        ProjectChanged?.Invoke(this, $"статус изменён на '{Status}'");
    }

    
    public void UpdateStatus(string status, string reason) {
        UpdateStatus(status);
        _logger.Log($"Причина: {reason}");
    }

    public virtual string GetDetails() =>
        $"ID:{ProjectId} | {ProjectName} | {Status} | Бюджет:{Budget} | Приоритет:{Priority} | Риск:{RiskLevel}";

    public virtual void AssignMember(TeamMember m) {
        Team.Add(m);
        _logger.Log($"'{ProjectName}': добавлен {m.Name}");
        ProjectChanged?.Invoke(this, $"добавлен участник {m.Name}");
    }

    
    public void SetPriority(string p) { Priority = p; _logger.Log($"'{ProjectName}': приоритет={p}"); }
    public void SetRisk(int r) { RiskLevel = r; }
    public int DaysSinceStart() => (DateTime.Now - StartDate).Days;
    public void AddTag(string tag) {
        if (Tags.Add(tag)) ProjectChanged?.Invoke(this, $"добавлен тег #{tag}");
    }

    
    string IExportable.ExportToXml() =>
        $"<Project><Id>{ProjectId}</Id><Name>{ProjectName}</Name><Status>{Status}</Status></Project>";
}


public class ProductProject : Project {
    public DateTime Deadline { get; set; }
    public string Version { get; set; } = "1.0";

    
    public string Platform { get; set; } = "";
    public int Users { get; set; } = 0;
    public int Progress { get; set; } = 0;

    public ProductProject(int id, string name, string status, DateTime date, ILogger logger)
        : base(id, name, status, logger) { Deadline = date; }

    public override void UpdateStatus(string s) {
        base.UpdateStatus(s);
        _logger.Log($"Дедлайн: {Deadline:d}");
    }
    public override string GetDetails() =>
        base.GetDetails() + $" | Версия:{Version} | Дедлайн:{Deadline:d} | Платформа:{Platform} | Пользователи:{Users} | Прогресс:{Progress}%";

    
    public void SetPlatform(string p) { Platform = p; }
    public void AddUsers(int n) { Users += n; }
    public void SetProgress(int p) { Progress = Math.Clamp(p, 0, 100); }
    public void Release(string v) { Version = v; _logger.Log($"Релиз {v}"); }
}


public class ResearchProject : Project {
    public decimal Funding { get; set; }
    public string Topic { get; set; }

    
    public string Area { get; set; } = "";
    public int Publications { get; set; } = 0;
    public string Laboratory { get; set; } = "";

    public ResearchProject(int id, string name, string status, decimal funding, string topic, ILogger logger)
        : base(id, name, status, logger) { Funding = funding; Topic = topic; }

    public override void AssignMember(TeamMember m) {
        if (!m.Skill.Contains("PhD")) { _logger.Log($"Ошибка: {m.Name} без PhD"); return; }
        base.AssignMember(m);
    }
    public override string GetDetails() =>
        base.GetDetails() + $" | Тема:{Topic} | Финансирование:{Funding} | Область:{Area} | Публикации:{Publications} | Лаборатория:{Laboratory}";

    
    public void SetArea(string a) { Area = a; }
    public void AddPublication() { Publications++; }
    public void SetLaboratory(string l) { Laboratory = l; }
}

public class AppliedResearchProject : ResearchProject, IReportable {
    public string Partner { get; set; }

    
    public string Patent { get; set; } = "Нет";
    public int Prototypes { get; set; } = 0;
    public string Contract { get; set; } = "Не подписан";

    public AppliedResearchProject(int id, string name, string status, decimal f, string t, string p, ILogger l)
        : base(id, name, status, f, t, l) { Partner = p; }

    public void GenerateReport() => _logger.Log($"Отчёт '{ProjectName}', партнёр: {Partner}");
    public override string GetDetails() =>
        base.GetDetails() + $" | Партнёр:{Partner} | Патент:{Patent} | Прототипы:{Prototypes} | Договор:{Contract}";

    
    public void FilePatent() { Patent = "Подан"; }
    public void CreatePrototype() { Prototypes++; }
    public void SignContract(string number) { Contract = number; }
}


public class InfrastructureProject : Project, IReportable {
    public string Period { get; set; }
    public string Location { get; set; }

  
    public string Type { get; set; } = "";
    public int Capacity { get; set; } = 0;
    public string Contractor { get; set; } = "";

    public InfrastructureProject(int id, string name, string status, string period, string loc, decimal b, ILogger l)
        : base(id, name, status, l, b) { Period = period; Location = loc; }

    public override string GetDetails() =>
        base.GetDetails() + $" | Срок:{Period} | Локация:{Location} | Тип:{Type} | Ёмкость:{Capacity} | Подрядчик:{Contractor}";
    public void GenerateReport() => _logger.Log($"Отчёт по инфраструктуре '{ProjectName}'");

    
    public void SetType(string t) { Type = t; }
    public void SetCapacity(int c) { Capacity = c; }
    public void SetContractor(string c) { Contractor = c; }
    public void Spend(decimal a) { Budget -= a; _logger.Log($"Потрачено {a}, остаток: {Budget}"); }
}


public class Organization {
    private Dictionary<int, Project> _projects = new();   
    private readonly ILogger _logger;
    public string Name { get; set; }

    public Organization(string name, ILogger l) { Name = name; _logger = l; }

    public void Add(Project p) { _projects[p.ProjectId] = p; _logger.Log($"Добавлен '{p.ProjectName}'"); }
    public void Assign(int id, TeamMember m) {
        if (_projects.TryGetValue(id, out var p)) p.AssignMember(m);
        else _logger.Log($"Проект с ID {id} не найден");
    }
    public void PrintAll() {
        Console.WriteLine($"\n{Name}");
        foreach (var p in _projects.Values) Console.WriteLine(p.GetDetails());
    }
    public void ReportAll() {
        foreach (var p in _projects.Values.OfType<IReportable>()) p.GenerateReport();
    }
}


public class ProjectRegistry<T> where T : Project {
    private List<T> _items = new();
    public void Add(T p) { _items.Add(p); }
    public int Count => _items.Count;

    public void FindByStatus(string s) {
        Console.WriteLine($"Поиск: {s}");
        var found = _items.Where(p => p.Status == s).ToList();
        if (!found.Any()) Console.WriteLine("Проектов не найдено.");
        else found.ForEach(p => Console.WriteLine(p.GetDetails()));
    }
}


var log = new ConsoleLogger();

Console.WriteLine("1. ProductProject (простое наследование, делегат и событие)");
var prod = new ProductProject(1, "Мобильное приложение", "В работе", DateTime.Now.AddMonths(6), log);
prod.ProjectChanged += (p, action) => Console.WriteLine($"[СОБЫТИЕ] '{p.ProjectName}': {action}");
prod.SetPriority("Высокий");
prod.SetPlatform("iOS");
prod.AddTag("мобильный");
prod.UpdateStatus("Тестирование");            
prod.UpdateStatus("Готов", "тесты пройдены"); 
prod.Release("2.0");
prod.AddUsers(1000);
prod.SetProgress(90);
Console.WriteLine(prod.GetDetails());

Console.WriteLine("\n2. AppliedResearchProject (сложное наследование)");
var res = new AppliedResearchProject(2, "ИИ в медицине", "Исследование", 1000000, "Нейросети", "МедТех", log);
res.ProjectChanged += (p, action) => Console.WriteLine($"[СОБЫТИЕ] '{p.ProjectName}': {action}");
res.SetArea("Диагностика");
res.SetLaboratory("Lab-7");
res.AddTag("ИИ");
res.AssignMember(new TeamMember("Иванов"));        
res.AssignMember(new TeamMember("Петров", "PhD")); 
res.AddPublication();
res.CreatePrototype();
res.FilePatent();
res.SignContract("Д-101");
Console.WriteLine(res.GetDetails());

Console.WriteLine("\n3. InfrastructureProject (интерфейсы вместо множественного наследования)");
var infra = new InfrastructureProject(3, "Дата-центр", "Строительство", "1 год", "Москва", 5000000, log);
infra.SetType("Серверная");
infra.SetCapacity(1000);
infra.SetContractor("СтройГрупп");
infra.AddTag("инфраструктура");
infra.Spend(1000000);
Console.WriteLine(infra.GetDetails());

Console.WriteLine("\n4. Organization (взаимодействие + Dictionary)");
var org = new Organization("Инновации", log);
org.Add(prod); org.Add(res); org.Add(infra);
org.Assign(1, new TeamMember("Сидоров"));
org.PrintAll();
org.ReportAll();

Console.WriteLine("\n5. Generic-класс");
var reg = new ProjectRegistry<Project>();
reg.Add(prod); reg.Add(res); reg.Add(infra);
Console.WriteLine($"Всего в реестре: {reg.Count}");
reg.FindByStatus("Исследование");

Console.WriteLine("\n6. HashSet: уникальные теги всех проектов");
var allTags = new HashSet<string>();
foreach (var p in new List<Project> { prod, res, infra })
    foreach (var t in p.Tags) allTags.Add(t);
Console.WriteLine(string.Join(", ", allTags));

Console.WriteLine("\n7. Явная реализация IExportable");
IExportable exp = prod;
Console.WriteLine(exp.ExportToXml());

Console.WriteLine("\n8. Доп. методы базового класса");
Console.WriteLine($"Дней с начала: {prod.DaysSinceStart()}");
prod.SetRisk(3);
Console.WriteLine(prod.GetDetails());

1. ProductProject (простое наследование, делегат и событие)
'Мобильное приложение': приоритет=Высокий
[СОБЫТИЕ] 'Мобильное приложение': добавлен тег #мобильный
'Мобильное приложение': статус -> Тестирование
[СОБЫТИЕ] 'Мобильное приложение': статус изменён на 'Тестирование'
Дедлайн: 5/20/2027
'Мобильное приложение': статус -> Готов
[СОБЫТИЕ] 'Мобильное приложение': статус изменён на 'Готов'
Дедлайн: 5/20/2027
Причина: тесты пройдены
Релиз 2.0
ID:1 | Мобильное приложение | Готов | Бюджет:0 | Приоритет:Высокий | Риск:1 | Версия:2.0 | Дедлайн:5/20/2027 | Платформа:iOS | Пользователи:1000 | Прогресс:90%

2. AppliedResearchProject (сложное наследование)
[СОБЫТИЕ] 'ИИ в медицине': добавлен тег #ИИ
Ошибка: Иванов без PhD
'ИИ в медицине': добавлен Петров
[СОБЫТИЕ] 'ИИ в медицине': добавлен участник Петров
ID:2 | ИИ в медицине | Исследование | Бюджет:0 | Приоритет:Средний | Риск:1 | Тема:Нейросети | Финансирование:1000000 | Область:Диагностика | Публикации:1 | Лаборатория:Lab-7 | Партнёр:МедТех 